# Part 21: Performance First Principles — Why Prefill and Decode Are Different Problems

This notebook derives one result, and that result explains almost every optimization in modern
LLM serving:

> **Prefill is compute-bound. Decode is memory-bandwidth-bound.**

Once you can derive that, a long list of techniques stops being a list of tricks and becomes
obvious consequences: why batching raises throughput almost for free, why speculative decoding
works at all, why GQA and MLA and quantization target exactly what they target, and why
FlashAttention is an IO optimization rather than an arithmetic one.

Everything here is arithmetic you can check, plus measurements on this machine. It is the
foundation for notebooks 22 (distributed training) and 23 (inference serving).

**What you'll build**
1. The hardware model: compute, bandwidth, and machine balance
2. **Arithmetic intensity** and the **roofline** model, measured on this CPU
3. Exact Transformer accounting: parameters, FLOPs, memory
4. The central derivation: prefill vs decode
5. Every optimization as a consequence
6. Latency metrics and the throughput-latency Pareto frontier
7. Memory budgets for training and inference

In [2]:
import math
import sys
import time

import torch
import matplotlib.pyplot as plt

sys.path.insert(0, '..')

torch.manual_seed(0)
print(f"torch {torch.__version__}, threads {torch.get_num_threads()}")

torch 2.9.1, threads 8


## 1. The hardware model

A processor has two headline numbers:

- **Compute throughput**, in FLOP/s — how fast it can multiply.
- **Memory bandwidth**, in bytes/s — how fast it can *feed* the multipliers.

Their ratio is the **machine balance**: how many floating-point operations the hardware can
perform per byte it reads.

```
machine_balance = peak_FLOPs_per_second / peak_bytes_per_second      [FLOP/byte]
```

This number is the whole story, and it is *much larger than intuition suggests*. Modern
accelerators are enormously better at arithmetic than at moving data — so any computation that
does less than a few hundred FLOPs per byte read is **wasting most of the chip**.

In [3]:
# Published specifications (dense bf16/fp16, no sparsity)
DEVICES = [
    # name,            TFLOP/s,  GB/s,   memory GB
    ("NVIDIA A100 80G",   312,   2039,   80),
    ("NVIDIA H100 SXM",   989,   3350,   80),
    ("NVIDIA H200",       989,   4800,  141),
    ("NVIDIA B200",      2250,   8000,  192),
    ("AMD MI300X",       1307,   5300,  192),
]

print(f"{'device':<20} {'TFLOP/s':>9} {'GB/s':>7} {'balance':>10}")
print("-" * 50)
for name, tflops, gbs, mem in DEVICES:
    balance = (tflops * 1e12) / (gbs * 1e9)
    print(f"{name:<20} {tflops:>9.0f} {gbs:>7.0f} {balance:>9.0f}")

print("\nRead that last column carefully. An H100 can do ~295 floating-point")
print("operations in the time it takes to read ONE byte from memory.")
print("\nA computation that performs fewer than 295 FLOPs per byte it reads")
print("leaves the compute units idle waiting on memory. That is the situation")
print("LLM decoding is in, and section 4 shows why.")

device                 TFLOP/s    GB/s    balance
--------------------------------------------------
NVIDIA A100 80G            312    2039       153
NVIDIA H100 SXM            989    3350       295
NVIDIA H200                989    4800       206
NVIDIA B200               2250    8000       281
AMD MI300X                1307    5300       247

Read that last column carefully. An H100 can do ~295 floating-point
operations in the time it takes to read ONE byte from memory.

A computation that performs fewer than 295 FLOPs per byte it reads
leaves the compute units idle waiting on memory. That is the situation
LLM decoding is in, and section 4 shows why.


## 2. Arithmetic intensity and the roofline

**Arithmetic intensity** is the property of an *algorithm*:

```
intensity = FLOPs performed / bytes moved      [FLOP/byte]
```

Compare it to the machine balance:
- `intensity > balance` → **compute-bound**. You are limited by arithmetic. Making the math
  cheaper (fewer FLOPs, faster kernels, lower precision arithmetic) helps.
- `intensity < balance` → **memory-bound**. You are limited by data movement. Making the math
  cheaper does *nothing*; you must move fewer bytes.

That distinction determines which optimizations can possibly help. Applying a compute
optimization to a memory-bound problem is wasted work — a mistake that is easy to make and
expensive to discover.

The **roofline** plots achievable performance against intensity: a diagonal (bandwidth-limited)
rising to a plateau (compute-limited), meeting at the machine balance.

In [4]:
def matmul_intensity(m, n, k, bytes_per_element=2):
    """
    Arithmetic intensity of C[m,n] = A[m,k] @ B[k,n].

    FLOPs: 2*m*n*k (one multiply and one add per element of the sum)
    Bytes: reading A and B, writing C
    """
    flops = 2 * m * n * k
    byte_count = (m * k + k * n + m * n) * bytes_per_element
    return flops / byte_count


print("Arithmetic intensity of some shapes:\n")
print(f"{'operation':<42} {'intensity':>10} {'on an H100'}")
print("-" * 74)
h100_balance = 989e12 / 3350e9

cases = [
    ("vector add: 1e6 + 1e6", 2 / (3 * 2 * 1) * 1),
    ("matrix-VECTOR: [4096,4096] @ [4096,1]", matmul_intensity(4096, 1, 4096)),
    ("small batch: [4096,4096] @ [4096,8]", matmul_intensity(4096, 8, 4096)),
    ("batch 64:    [4096,4096] @ [4096,64]", matmul_intensity(4096, 64, 4096)),
    ("batch 512:   [4096,4096] @ [4096,512]", matmul_intensity(4096, 512, 4096)),
    ("square:      [4096,4096] @ [4096,4096]", matmul_intensity(4096, 4096, 4096)),
]
for label, intensity in cases:
    verdict = "compute-bound" if intensity > h100_balance else "MEMORY-bound"
    print(f"{label:<42} {intensity:>10.1f}   {verdict}")

print(f"\n(H100 balance = {h100_balance:.0f} FLOP/byte)")
print("\nNote the progression. A matrix-VECTOR product has intensity ~2 -- it")
print("reads a huge weight matrix to do almost no arithmetic. Widening that")
print("vector into a batch raises intensity almost linearly, and somewhere")
print("around batch 256-512 the operation finally becomes compute-bound.")
print("\nRemember that number. It is the single most important fact about")
print("serving LLMs, and section 4 explains why.")

Arithmetic intensity of some shapes:

operation                                   intensity on an H100
--------------------------------------------------------------------------
vector add: 1e6 + 1e6                             0.3   MEMORY-bound
matrix-VECTOR: [4096,4096] @ [4096,1]             1.0   MEMORY-bound
small batch: [4096,4096] @ [4096,8]               8.0   MEMORY-bound
batch 64:    [4096,4096] @ [4096,64]             62.1   MEMORY-bound
batch 512:   [4096,4096] @ [4096,512]           409.6   compute-bound
square:      [4096,4096] @ [4096,4096]         1365.3   compute-bound

(H100 balance = 295 FLOP/byte)

Note the progression. A matrix-VECTOR product has intensity ~2 -- it
reads a huge weight matrix to do almost no arithmetic. Widening that
vector into a batch raises intensity almost linearly, and somewhere
around batch 256-512 the operation finally becomes compute-bound.

Remember that number. It is the single most important fact about
serving LLMs, and section 4 explain

### Measuring the real roofline on this machine

Specifications are optimistic. Let's measure what this CPU actually achieves, by timing matrix
multiplies across a range of shapes and computing achieved FLOP/s against intensity.

In [5]:
def time_matmul(m, n, k, repeats=None, dtype=torch.float32):
    """Measured FLOP/s for one matmul shape."""
    a = torch.randn(m, k, dtype=dtype)
    b = torch.randn(k, n, dtype=dtype)

    if repeats is None:
        # More repeats for small problems, to get out of timer noise
        repeats = max(3, min(200, int(2e8 / (m * n * k + 1))))

    torch.matmul(a, b)                      # warm up
    start = time.perf_counter()
    for _ in range(repeats):
        torch.matmul(a, b)
    elapsed = time.perf_counter() - start

    flops = 2 * m * n * k * repeats
    return flops / elapsed


print("Measuring achieved throughput (fp32) ...\n")
measurements = []
K = 1024
for n in (1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024):
    achieved = time_matmul(K, n, K)
    intensity = matmul_intensity(K, n, K, bytes_per_element=4)
    measurements.append((n, intensity, achieved))

peak = max(m[2] for m in measurements)
print(f"{'batch n':>9} {'intensity':>11} {'GFLOP/s':>10} {'% of measured peak':>20}")
print("-" * 54)
for n, intensity, achieved in measurements:
    print(f"{n:>9} {intensity:>11.1f} {achieved/1e9:>10.1f} "
          f"{100*achieved/peak:>19.0f}%")

Measuring achieved throughput (fp32) ...

  batch n   intensity    GFLOP/s   % of measured peak
------------------------------------------------------
        1         0.5      183.6                   8%
        2         1.0       42.9                   2%
        4         2.0       80.7                   3%
        8         3.9      180.7                   8%
       16         7.8      364.5                  16%
       32        15.1      585.8                  25%
       64        28.4      830.2                  36%
      128        51.2     1311.2                  57%
      256        85.3     2317.4                 100%
      512       128.0     1914.0                  83%
     1024       170.7     1746.5                  75%


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

intensities = [m[1] for m in measurements]
achieved = [m[2] / 1e9 for m in measurements]

axes[0].loglog(intensities, achieved, 'o-', color='#2E86AB', lw=2,
               label='measured (this machine)')

# Fit the roofline from the extremes of the measured data
measured_peak = peak / 1e9
# Bandwidth implied by the lowest-intensity point
implied_bw = measurements[0][2] / measurements[0][1] / 1e9
x = torch.logspace(-1, 3, 100)
roof = torch.minimum(torch.full_like(x, measured_peak), implied_bw * x)
axes[0].loglog(x, roof, '--', color='#C73E1D', lw=1.5, label='roofline')
axes[0].axvline(measured_peak / implied_bw, color='gray', ls=':',
                label=f'balance ~{measured_peak/implied_bw:.0f} FLOP/byte')
axes[0].set_xlabel('arithmetic intensity (FLOP/byte)')
axes[0].set_ylabel('achieved GFLOP/s')
axes[0].set_title('Measured roofline')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3, which='both')

batches = [m[0] for m in measurements]
axes[1].semilogx(batches, [100 * m[2] / peak for m in measurements], 'o-',
                 color='#3B7A57', lw=2, base=2)
axes[1].set_xlabel('batch size n (columns of B)')
axes[1].set_ylabel('% of measured peak throughput')
axes[1].set_title('Batching turns a memory-bound op into a compute-bound one')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"measured peak: {measured_peak:.0f} GFLOP/s")
print(f"implied bandwidth: {implied_bw:.1f} GB/s")
print(f"implied machine balance: {measured_peak/implied_bw:.0f} FLOP/byte")
print("\nThe left plot is the roofline: a bandwidth-limited diagonal rising to a")
print("compute-limited ceiling. The right plot says the same thing in the units")
print("that matter for serving -- batch size is what moves you along the roof.")
print("\n(CPU numbers are far smaller than a GPU's, and caches make the picture")
print("messier. The SHAPE is what transfers.)")

measured peak: 2317 GFLOP/s
implied bandwidth: 368.0 GB/s
implied machine balance: 6 FLOP/byte

The left plot is the roofline: a bandwidth-limited diagonal rising to a
compute-limited ceiling. The right plot says the same thing in the units
that matter for serving -- batch size is what moves you along the roof.

(CPU numbers are far smaller than a GPU's, and caches make the picture
messier. The SHAPE is what transfers.)


## 3. Exact Transformer accounting

Now the model side. These formulas are worth memorizing; they answer most capacity-planning
questions without a profiler.

**Parameters.** Per layer, with `d = d_model` and `d_ff = 4d`:
- Attention: `W_q, W_k, W_v, W_o` → `4d²`
- Feed-forward: two matrices of `d × 4d` → `8d²`
- Total per layer: **`12d²`**

For `L` layers: `12Ld²`, plus `V·d` for embeddings (which dominates only in small models).

**Forward FLOPs.** Each parameter participates in one multiply-add per token, and a multiply-add
is 2 FLOPs:
```
forward ≈ 2 · N · P            (N tokens, P parameters)
```

**Training FLOPs.** The backward pass computes gradients with respect to both inputs and weights,
costing about twice the forward pass:
```
training ≈ 6 · N · P           (2 forward + 4 backward)
```

That `6ND` is the formula behind every scaling-law and compute-budget calculation, including
Chinchilla's.

**The attention term.** `QKᵀ` and `attn·V` add `4·T²·d` per layer, which is *not* proportional to
parameters. It is negligible for short sequences and dominant for long ones.

In [7]:
def model_accounting(d_model, num_layers, vocab_size, seq_len, d_ff_mult=4):
    """Parameters and per-token FLOPs, with the attention term separated."""
    per_layer = (4 + 2 * d_ff_mult) * d_model ** 2
    params = num_layers * per_layer + vocab_size * d_model

    # Parameter-proportional FLOPs, per token
    dense_flops = 2 * params
    # Attention's quadratic term, amortized per token: 4*T^2*d per layer / T
    attn_flops = num_layers * 4 * seq_len * d_model

    return {
        'params': params,
        'dense_flops_per_token': dense_flops,
        'attn_flops_per_token': attn_flops,
        'attn_share': attn_flops / (dense_flops + attn_flops),
    }


MODELS = [
    # name,           d_model, layers, vocab
    ("GPT-2 small",       768,     12,  50257),
    ("Llama-3-8B",       4096,     32, 128256),
    ("Llama-3-70B",      8192,     80, 128256),
]

print(f"{'model':<16} {'d':>6} {'L':>4} {'params (formula)':>18} {'actual':>9}")
print("-" * 60)
actuals = {"GPT-2 small": "124M", "Llama-3-8B": "8.0B", "Llama-3-70B": "70B"}
for name, d, L, V in MODELS:
    info = model_accounting(d, L, V, 2048)
    print(f"{name:<16} {d:>6} {L:>4} {info['params']/1e9:>17.2f}B "
          f"{actuals[name]:>9}")

print("\nThe 12Ld^2 estimate lands close for the Llama models. GPT-2 small is")
print("off because its vocabulary embedding is a large share of a small model,")
print("and Llama uses SwiGLU (3 matrices) with a non-4x d_ff -- so treat the")
print("formula as a sizing tool, not an exact count.")

model                 d    L   params (formula)    actual
------------------------------------------------------------
GPT-2 small         768   12              0.12B      124M
Llama-3-8B         4096   32              6.97B      8.0B
Llama-3-70B        8192   80             65.48B       70B

The 12Ld^2 estimate lands close for the Llama models. GPT-2 small is
off because its vocabulary embedding is a large share of a small model,
and Llama uses SwiGLU (3 matrices) with a non-4x d_ff -- so treat the
formula as a sizing tool, not an exact count.


In [8]:
print("\nWhen does the attention term start to matter? (Llama-3-8B shape)\n")
print(f"{'seq_len':>9} {'dense GFLOP/tok':>17} {'attn GFLOP/tok':>16} {'attn share':>12}")
print("-" * 60)
for T in (512, 2048, 8192, 32768, 131072):
    info = model_accounting(4096, 32, 128256, T)
    print(f"{T:>9} {info['dense_flops_per_token']/1e9:>17.2f} "
          f"{info['attn_flops_per_token']/1e9:>16.2f} {info['attn_share']:>11.1%}")

print("\nUnder ~8k tokens attention is a minority of the compute -- the FFN and")
print("projections dominate. Past 32k it takes over. This is why long context")
print("changed which optimizations matter (notebooks 15 and 16), and why")
print("FlashAttention arrived exactly when context windows started growing.")


When does the attention term start to matter? (Llama-3-8B shape)

  seq_len   dense GFLOP/tok   attn GFLOP/tok   attn share
------------------------------------------------------------
      512             13.94             0.27        1.9%
     2048             13.94             1.07        7.2%
     8192             13.94             4.29       23.6%
    32768             13.94            17.18       55.2%
   131072             13.94            68.72       83.1%

Under ~8k tokens attention is a minority of the compute -- the FFN and
projections dominate. Past 32k it takes over. This is why long context
changed which optimizations matter (notebooks 15 and 16), and why
FlashAttention arrived exactly when context windows started growing.


## 4. The central result

Now the derivation everything else depends on. We compute the arithmetic intensity of the two
phases of inference.

### Prefill

The prompt has `T` tokens and they are processed **together**. Weights are read once and used
for all `T` tokens:

```
FLOPs ≈ 2 · P · T
bytes ≈ P · bytes_per_param        (each weight read once)
intensity ≈ 2T / bytes_per_param
```

With `T = 1000` and fp16, intensity ≈ **1000 FLOP/byte** — far above any machine balance.
**Compute-bound.**

### Decode

One token at a time. The *same weights* are read, to do `1/T` of the work:

```
FLOPs ≈ 2 · P · B                  (B = batch size)
bytes ≈ P · bytes_per_param        (still every weight, once)
intensity ≈ 2B / bytes_per_param
```

With `B = 1` and fp16, intensity ≈ **1 FLOP/byte**. Against an H100's 295, that is using
**under half a percent** of the available arithmetic. Utterly **memory-bound**.

The asymmetry is stark: prefill and decode run the same weights through the same kernels and sit
at *opposite ends of the roofline*.

In [9]:
def phase_intensity(num_tokens, bytes_per_param=2):
    """Arithmetic intensity of a phase processing `num_tokens` tokens at once."""
    return 2 * num_tokens / bytes_per_param


print("Arithmetic intensity by phase (fp16 weights):\n")
print(f"{'phase':<34} {'tokens at once':>15} {'intensity':>11} {'verdict':>16}")
print("-" * 80)
for label, n in (
    ("prefill, 2048-token prompt", 2048),
    ("prefill, 128-token prompt", 128),
    ("decode, batch 1", 1),
    ("decode, batch 8", 8),
    ("decode, batch 64", 64),
    ("decode, batch 256", 256),
    ("decode, batch 512", 512),
):
    intensity = phase_intensity(n)
    verdict = "compute-bound" if intensity > h100_balance else "memory-bound"
    print(f"{label:<34} {n:>15} {intensity:>11.0f} {verdict:>16}")

crossover = h100_balance * 2 / 2
print(f"\nDecode becomes compute-bound at batch ~{crossover:.0f} on an H100.")
print("\nThat single number drives serving architecture. Below it, you are")
print("paying for memory traffic and getting arithmetic for free -- so the")
print("goal of a serving system is to GET THE BATCH SIZE UP. Notebook 23 is")
print("largely about how.")

Arithmetic intensity by phase (fp16 weights):

phase                               tokens at once   intensity          verdict
--------------------------------------------------------------------------------
prefill, 2048-token prompt                    2048        2048    compute-bound
prefill, 128-token prompt                      128         128     memory-bound
decode, batch 1                                  1           1     memory-bound
decode, batch 8                                  8           8     memory-bound
decode, batch 64                                64          64     memory-bound
decode, batch 256                              256         256     memory-bound
decode, batch 512                              512         512    compute-bound

Decode becomes compute-bound at batch ~295 on an H100.

That single number drives serving architecture. Below it, you are
paying for memory traffic and getting arithmetic for free -- so the
goal of a serving system is to GET THE BA

In [10]:
# What decode's memory-boundness costs, in achievable tokens per second
def decode_ceiling(params, batch_size, bandwidth_gbs, bytes_per_param=2):
    """
    Upper bound on decode throughput from bandwidth alone.

    Every step must read every weight, so time per step >= bytes / bandwidth.
    """
    bytes_per_step = params * bytes_per_param
    steps_per_second = (bandwidth_gbs * 1e9) / bytes_per_step
    return steps_per_second, steps_per_second * batch_size


print("\nDecode throughput ceiling on an H100 (3350 GB/s), 8B parameters:\n")
print(f"{'batch':>7} {'steps/s':>10} {'tokens/s total':>16} {'tokens/s each':>15}")
print("-" * 52)
for B in (1, 8, 32, 128, 256):
    steps, total = decode_ceiling(8e9, B, 3350)
    print(f"{B:>7} {steps:>10.0f} {total:>16,.0f} {steps:>15.0f}")

print("\nThe steps/s column is CONSTANT -- reading the weights takes the same")
print("time regardless of batch size, because it is the same weights. So every")
print("extra sequence in the batch is nearly free throughput, and per-user")
print("latency does not degrade until you become compute-bound.")
print("\nThis is the economics of LLM serving in one table.")


Decode throughput ceiling on an H100 (3350 GB/s), 8B parameters:

  batch    steps/s   tokens/s total   tokens/s each
----------------------------------------------------
      1        209              209             209
      8        209            1,675             209
     32        209            6,700             209
    128        209           26,800             209
    256        209           53,600             209

The steps/s column is CONSTANT -- reading the weights takes the same
time regardless of batch size, because it is the same weights. So every
extra sequence in the batch is nearly free throughput, and per-user
latency does not degrade until you become compute-bound.

This is the economics of LLM serving in one table.


## 5. Every optimization, as a consequence

With the derivation in hand, a long list of techniques collapses into a short one.

| Technique | What it changes | Why it works |
|---|---|---|
| **Continuous batching** (NB 23) | Raises effective batch size | Moves decode right along the roofline; nearly free throughput |
| **GQA / MQA** (NB 11, 16) | Fewer KV bytes to read | Decode is bandwidth-bound, and the cache is part of the traffic |
| **MLA** (NB 16) | Cache 57x smaller, more FLOPs | *Deliberately* trades compute for bandwidth — the right direction |
| **Weight quantization** (NB 19) | Fewer bytes per weight | Directly divides decode's bottleneck |
| **KV cache quantization** (NB 15) | Fewer cache bytes | Same reason |
| **Speculative decoding** (NB 23) | Verifies `k` tokens in one pass | Uses idle *compute* to buy serial steps; only works because decode is memory-bound |
| **FlashAttention** (NB 11) | Never materializes the score matrix | An IO optimization. Same FLOPs, far fewer bytes |
| **Chunked prefill** (NB 23) | Mixes prefill and decode in one batch | Fills a compute-bound phase's idle bandwidth with a memory-bound one's needs |

Speculative decoding deserves a moment because it is the least intuitive. Running a draft model
and verifying `k` tokens in a single forward pass costs `k×` the FLOPs of one step — but decode
was only using ~0.5% of the chip's arithmetic, so those FLOPs were **free**. You convert wasted
compute into fewer sequential memory-bound steps. On a compute-bound workload it would be
pointless.

In [11]:
# Speculative decoding's arithmetic, made concrete
def speculative_speedup(acceptance_rate, gamma):
    """
    Expected tokens per verification step.

    With acceptance probability p per draft token and gamma drafted, the
    expected number accepted is a truncated geometric series, plus one token
    that the verifier always produces.
    """
    if acceptance_rate >= 1.0:
        return gamma + 1
    return (1 - acceptance_rate ** (gamma + 1)) / (1 - acceptance_rate)


print("Speculative decoding: expected tokens per target-model forward pass\n")
print(f"{'accept rate':>12} " + " ".join(f"{'g=' + str(g):>7}" for g in (1, 2, 4, 8)))
print("-" * 46)
for p in (0.5, 0.7, 0.8, 0.9):
    row = " ".join(f"{speculative_speedup(p, g):>7.2f}" for g in (1, 2, 4, 8))
    print(f"{p:>12.2f} {row}")

print("\nAt 80% acceptance and 4 drafted tokens, each target forward pass yields")
print(f"{speculative_speedup(0.8, 4):.2f} tokens instead of 1 -- roughly a 3x latency")
print("improvement, paid for entirely in FLOPs that were being wasted anyway.")
print("\nNote the diminishing returns in gamma: drafting more only helps if the")
print("draft model stays accurate that far ahead.")

Speculative decoding: expected tokens per target-model forward pass

 accept rate     g=1     g=2     g=4     g=8
----------------------------------------------
        0.50    1.50    1.75    1.94    2.00
        0.70    1.70    2.19    2.77    3.20
        0.80    1.80    2.44    3.36    4.33
        0.90    1.90    2.71    4.10    6.13

At 80% acceptance and 4 drafted tokens, each target forward pass yields
3.36 tokens instead of 1 -- roughly a 3x latency
improvement, paid for entirely in FLOPs that were being wasted anyway.

Note the diminishing returns in gamma: drafting more only helps if the
draft model stays accurate that far ahead.


## 6. Latency metrics

Serving has three numbers that users actually feel, and they are governed by different phases.

- **TTFT** (time to first token) — dominated by **prefill**. Scales with prompt length.
- **TPOT** (time per output token), also called ITL (inter-token latency) — dominated by
  **decode**. Roughly constant per token.
- **Throughput** — total tokens per second across all users.

And a structural tension: **throughput and latency trade off against each other.** Larger batches
raise throughput (better roofline position) but each request waits longer in the queue and in
each step. There is no single best operating point, only a Pareto frontier and an SLO to pick
from it.

In [12]:
def request_latency(prompt_tokens, output_tokens, params, batch_size,
                    tflops, bandwidth_gbs, bytes_per_param=2):
    """
    Rough end-to-end latency, using the bound from whichever resource limits
    each phase.
    """
    # Prefill: compute-bound
    prefill_flops = 2 * params * prompt_tokens
    ttft = prefill_flops / (tflops * 1e12)

    # Decode: bandwidth-bound, one weight sweep per step regardless of batch
    bytes_per_step = params * bytes_per_param
    tpot = bytes_per_step / (bandwidth_gbs * 1e9)
    decode_time = tpot * output_tokens

    return ttft, tpot, ttft + decode_time


PARAMS, TFLOPS, BW = 8e9, 989, 3350

print("Llama-3-8B on one H100 (idealized, ignoring overheads):\n")
print(f"{'prompt':>8} {'output':>8} {'TTFT':>9} {'TPOT':>9} {'total':>9} {'tok/s/user':>12}")
print("-" * 62)
for prompt, output in ((128, 256), (2048, 256), (8192, 256), (32768, 512)):
    ttft, tpot, total = request_latency(prompt, output, PARAMS, 1, TFLOPS, BW)
    print(f"{prompt:>8} {output:>8} {ttft*1000:>8.0f}ms {tpot*1000:>8.1f}ms "
          f"{total:>8.2f}s {1/tpot:>12.0f}")

print("\nTTFT grows linearly with prompt length; TPOT does not depend on it at")
print("all. A user with a 32k prompt waits a long time for the first token and")
print("then sees the same streaming speed as everyone else.")
print("\nThat asymmetry is why prefill and decode are increasingly run on")
print("SEPARATE machines (notebook 23's disaggregation): one is compute-hungry,")
print("the other bandwidth-hungry, and mixing them means neither runs well.")

Llama-3-8B on one H100 (idealized, ignoring overheads):

  prompt   output      TTFT      TPOT     total   tok/s/user
--------------------------------------------------------------
     128      256        2ms      4.8ms     1.22s          209
    2048      256       33ms      4.8ms     1.26s          209
    8192      256      133ms      4.8ms     1.36s          209
   32768      512      530ms      4.8ms     2.98s          209

TTFT grows linearly with prompt length; TPOT does not depend on it at
all. A user with a 32k prompt waits a long time for the first token and
then sees the same streaming speed as everyone else.

That asymmetry is why prefill and decode are increasingly run on
SEPARATE machines (notebook 23's disaggregation): one is compute-hungry,
the other bandwidth-hungry, and mixing them means neither runs well.


In [13]:
# The throughput-latency Pareto frontier
def frontier_point(batch_size, params=8e9, tflops=989, bandwidth_gbs=3350,
                   bytes_per_param=2):
    """
    Per-step time and aggregate throughput at a given decode batch size.

    Below the compute crossover the step time is bandwidth-limited and constant;
    above it, compute-limited and growing with batch size.
    """
    bytes_per_step = params * bytes_per_param
    bandwidth_time = bytes_per_step / (bandwidth_gbs * 1e9)
    compute_time = (2 * params * batch_size) / (tflops * 1e12)
    step_time = max(bandwidth_time, compute_time)
    return step_time, batch_size / step_time


batch_sizes = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]
points = [frontier_point(b) for b in batch_sizes]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot([p[1] for p in points], [p[0] * 1000 for p in points], 'o-',
             color='#2E86AB', lw=2)
for b, (step, thr) in zip(batch_sizes, points):
    if b in (1, 32, 256, 1024):
        axes[0].annotate(f'batch {b}', (thr, step * 1000), fontsize=8,
                         xytext=(5, 5), textcoords='offset points')
axes[0].set_xlabel('throughput (tokens/s, all users)')
axes[0].set_ylabel('per-token latency (ms)')
axes[0].set_title('Throughput vs latency: pick a point, not an optimum')
axes[0].set_xscale('log')
axes[0].grid(alpha=0.3)

intensity = [2 * b / 2 for b in batch_sizes]
utilization = [
    100 * (2 * 8e9 * b / (p[0] * 989e12)) for b, p in zip(batch_sizes, points)
]
axes[1].semilogx(batch_sizes, utilization, 'o-', color='#C73E1D', lw=2, base=2)
axes[1].axvline(h100_balance, ls=':', color='gray',
                label=f'compute crossover (~{h100_balance:.0f})')
axes[1].set_xlabel('decode batch size')
axes[1].set_ylabel('% of peak FLOP/s achieved')
axes[1].set_title('Compute utilization during decode')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

step1, thr1 = points[0]
step_big, thr_big = points[batch_sizes.index(256)]
print(f"batch 1:   {thr1:>8.0f} tok/s total, {step1*1000:>5.1f} ms/token")
print(f"batch 256: {thr_big:>8.0f} tok/s total, {step_big*1000:>5.1f} ms/token")
print(f"\n{thr_big/thr1:.0f}x the throughput at "
      f"{step_big/step1:.1f}x the per-token latency.")
print("Below the crossover, batching is nearly free. That is why every serving")
print("system works hard to keep batches full.")

batch 1:        209 tok/s total,   4.8 ms/token
batch 256:    53600 tok/s total,   4.8 ms/token

256x the throughput at 1.0x the per-token latency.
Below the crossover, batching is nearly free. That is why every serving
system works hard to keep batches full.


## 7. Memory budgets

The other planning question: what fits? Training and inference have very different answers, and
the training answer surprises people.

**Training** needs, per parameter:
- the parameter itself (2 bytes in bf16)
- a gradient (2–4 bytes)
- AdamW's two moment estimates, **in fp32** (8 bytes)
- often an fp32 master copy of the weights (4 bytes)

That is **16–18 bytes per parameter** before a single activation. Optimizer state, not weights,
is usually what does not fit — which is exactly what ZeRO/FSDP shards (notebook 22).

**Inference** needs weights plus the KV cache. The cache grows with batch and sequence length
while the weights are shared, so at scale the cache dominates (notebook 15).

In [14]:
def training_memory(params, bytes_per_param=2, optimizer='adamw',
                    master_weights=True):
    """Per-parameter training memory, broken down."""
    components = {'weights': params * bytes_per_param,
                  'gradients': params * bytes_per_param}
    if optimizer == 'adamw':
        components['optimizer (2 fp32 moments)'] = params * 8
    elif optimizer == 'sgd':
        components['optimizer (momentum)'] = params * 4
    if master_weights:
        components['fp32 master weights'] = params * 4
    return components


print("Training memory for an 8B model (bf16 + AdamW), excluding activations:\n")
components = training_memory(8e9)
total = sum(components.values())
print(f"{'component':<32} {'GB':>8} {'share':>8}")
print("-" * 52)
for name, size in components.items():
    print(f"{name:<32} {size/1e9:>8.1f} {size/total:>7.0%}")
print("-" * 52)
print(f"{'TOTAL':<32} {total/1e9:>8.1f}")
print(f"\n{total/8e9:.0f} bytes per parameter. The weights are only "
      f"{components['weights']/total:.0%} of it.")
print(f"An 8B model needs ~{total/1e9:.0f}GB before activations -- it does not fit")
print("on an 80GB card with any real batch size. Hence FSDP.")

Training memory for an 8B model (bf16 + AdamW), excluding activations:

component                              GB    share
----------------------------------------------------
weights                              16.0     12%
gradients                            16.0     12%
optimizer (2 fp32 moments)           64.0     50%
fp32 master weights                  32.0     25%
----------------------------------------------------
TOTAL                               128.0

16 bytes per parameter. The weights are only 12% of it.
An 8B model needs ~128GB before activations -- it does not fit
on an 80GB card with any real batch size. Hence FSDP.


In [15]:
print("\nActivation memory, and what recomputation buys:\n")

def activation_memory(batch, seq_len, d_model, num_layers, num_heads,
                      bytes_per_element=2, checkpointing=False):
    """
    Activations stored for the backward pass.

    Without checkpointing, roughly a dozen tensors of (B,T,d) per layer plus
    the attention score matrix. With checkpointing, only the layer boundaries
    are kept and the interior is recomputed.
    """
    per_layer = 12 * batch * seq_len * d_model * bytes_per_element
    scores = num_heads * batch * seq_len ** 2 * bytes_per_element
    if checkpointing:
        # Store one tensor per layer boundary; recompute the rest
        return num_layers * batch * seq_len * d_model * bytes_per_element
    return num_layers * (per_layer + scores)


print(f"{'batch':>7} {'seq':>7} {'no checkpoint':>15} {'checkpointed':>14} {'ratio':>8}")
print("-" * 56)
for batch, seq in ((1, 2048), (8, 2048), (8, 8192), (1, 32768)):
    plain = activation_memory(batch, seq, 4096, 32, 32)
    ckpt = activation_memory(batch, seq, 4096, 32, 32, checkpointing=True)
    print(f"{batch:>7} {seq:>7} {plain/1e9:>14.1f}G {ckpt/1e9:>13.1f}G "
          f"{plain/ckpt:>8.0f}x")

print("\nActivations dwarf the weights at any real batch size and sequence")
print("length. Gradient checkpointing trades ~30% extra compute for a large")
print("memory reduction, and at long context it is not optional.")
print("\nNote the seq^2 attention-score term -- this is exactly what")
print("FlashAttention removes by never materializing that matrix.")


Activation memory, and what recomputation buys:

  batch     seq   no checkpoint   checkpointed    ratio
--------------------------------------------------------
      1    2048           15.0G           0.5G       28x
      8    2048          120.3G           4.3G       28x
      8    8192         1305.7G          17.2G       76x
      1   32768         2302.1G           8.6G      268x

Activations dwarf the weights at any real batch size and sequence
length. Gradient checkpointing trades ~30% extra compute for a large
memory reduction, and at long context it is not optional.

Note the seq^2 attention-score term -- this is exactly what
FlashAttention removes by never materializing that matrix.


In [16]:
print("\nInference memory for an 8B model (fp16 weights, GQA 8 kv heads):\n")
print(f"{'batch':>7} {'seq':>8} {'weights':>10} {'KV cache':>11} {'total':>9} {'cache share':>13}")
print("-" * 66)
weights_gb = 8e9 * 2 / 1e9
for batch, seq in ((1, 2048), (32, 2048), (32, 32768), (128, 8192)):
    cache = 2 * 32 * 8 * 128 * seq * batch * 2 / 1e9
    print(f"{batch:>7} {seq:>8} {weights_gb:>9.1f}G {cache:>10.1f}G "
          f"{weights_gb+cache:>8.1f}G {cache/(weights_gb+cache):>12.0%}")

print("\nAt serving scale the cache is the majority of memory. It is also what")
print("limits how large a batch you can run -- and section 4 showed that batch")
print("size is what determines throughput. So cache size directly caps")
print("throughput, which is why PagedAttention (notebook 23) mattered so much.")


Inference memory for an 8B model (fp16 weights, GQA 8 kv heads):

  batch      seq    weights    KV cache     total   cache share
------------------------------------------------------------------
      1     2048      16.0G        0.3G     16.3G           2%
     32     2048      16.0G        8.6G     24.6G          35%
     32    32768      16.0G      137.4G    153.4G          90%
    128     8192      16.0G      137.4G    153.4G          90%

At serving scale the cache is the majority of memory. It is also what
limits how large a batch you can run -- and section 4 showed that batch
size is what determines throughput. So cache size directly caps
throughput, which is why PagedAttention (notebook 23) mattered so much.


## 8. MFU — are you using the machine?

**Model FLOPs Utilization** is the honest efficiency metric: achieved model FLOPs divided by the
hardware's peak.

```
MFU = (6 · tokens · params / elapsed_seconds) / peak_FLOPs
```

Note it counts only the *useful* model FLOPs, so recomputation from gradient checkpointing does
not flatter the number.

Realistic targets: **40–55%** for well-tuned large-scale training. Above 60% is exceptional.
Below 25% means something is wrong — usually communication stalls or a pipeline bubble
(notebook 22).

For *inference decode*, MFU is intrinsically terrible (well under 5%) and that is fine —
section 4 explains why. Use **MBU** (Model Bandwidth Utilization) instead: achieved bytes/s over
peak bandwidth. That is the resource decode actually consumes.

In [17]:
def mfu(tokens, params, seconds, peak_tflops):
    return (6 * tokens * params / seconds) / (peak_tflops * 1e12)


def mbu(params, tokens_per_second, batch_size, peak_bandwidth_gbs,
        bytes_per_param=2):
    """Bandwidth utilization during decode."""
    steps_per_second = tokens_per_second / batch_size
    bytes_per_second = steps_per_second * params * bytes_per_param
    return bytes_per_second / (peak_bandwidth_gbs * 1e9)


print("Training MFU examples (8B model, one H100 at 989 TFLOP/s):\n")
print(f"{'tokens/s':>12} {'MFU':>8}  assessment")
print("-" * 46)
for tps in (2000, 5000, 10000, 15000):
    value = mfu(tps, 8e9, 1.0, 989)
    note = ("something is broken" if value < 0.15
            else "poor" if value < 0.3
            else "acceptable" if value < 0.45
            else "good")
    print(f"{tps:>12,} {value:>7.1%}  {note}")

print("\nDecode, same hardware -- MFU vs MBU:\n")
print(f"{'batch':>7} {'tok/s':>9} {'MFU':>8} {'MBU':>8}")
print("-" * 36)
for B in (1, 32, 256):
    steps, total = decode_ceiling(8e9, B, 3350)
    print(f"{B:>7} {total:>9,.0f} {mfu(total, 8e9, 1.0, 989):>7.1%} "
          f"{mbu(8e9, total, B, 3350):>7.1%}")

print("\nAt batch 1, MFU is a fraction of a percent while MBU is ~100%. The")
print("machine is fully utilized -- just not in the dimension MFU measures.")
print("Reporting MFU for decode is a category error; report MBU.")

Training MFU examples (8B model, one H100 at 989 TFLOP/s):

    tokens/s      MFU  assessment
----------------------------------------------
       2,000    9.7%  something is broken
       5,000   24.3%  poor
      10,000   48.5%  good
      15,000   72.8%  good

Decode, same hardware -- MFU vs MBU:

  batch     tok/s      MFU      MBU
------------------------------------
      1       209    1.0%  100.0%
     32     6,700   32.5%  100.0%
    256    53,600  260.1%  100.0%

At batch 1, MFU is a fraction of a percent while MBU is ~100%. The
machine is fully utilized -- just not in the dimension MFU measures.
Reporting MFU for decode is a category error; report MBU.


## Summary

One derivation, many consequences.

**The hardware model.** Machine balance = peak FLOP/s ÷ peak bytes/s. On an H100 that is ~295
FLOP/byte: the chip can do 295 arithmetic operations in the time it reads one byte. Any
computation below that ratio leaves compute idle.

**Arithmetic intensity** = FLOPs ÷ bytes moved. Above the balance you are compute-bound (make
the math cheaper); below it you are memory-bound (move fewer bytes). Applying the wrong class of
optimization accomplishes nothing.

**The accounting.** `12Ld²` parameters per layer-stack; `2NP` forward FLOPs; `6NP` training
FLOPs (2 forward + 4 backward); plus a `4T²d` attention term that is negligible below ~8k tokens
and dominant past 32k.

**The central result.** Prefill processes `T` tokens per weight-read, giving intensity `≈2T` —
compute-bound. Decode processes `B` tokens per weight-read, giving intensity `≈2B` —
memory-bound until batch ~300 on an H100. **Same weights, same kernels, opposite ends of the
roofline.**

**Which explains everything else.** Batching is nearly free throughput below the crossover.
Speculative decoding converts idle FLOPs into fewer sequential steps. GQA, MLA, and quantization
all reduce bytes, which is the actual bottleneck. FlashAttention is an IO optimization.
Disaggregating prefill from decode follows from them wanting different hardware.

**Memory.** Training costs 16–18 bytes per parameter *before activations* — the weights are a
minority and the optimizer state is what fails to fit. Inference is weights plus a KV cache that
scales with batch × length, so the cache caps batch size, which caps throughput.

**Measure the right thing.** MFU for training (target 40–55%); MBU for decode, where MFU is
intrinsically near zero and that is correct.

### Key Takeaways

1. **Machine balance is a few hundred FLOP/byte.** Modern accelerators are starved for data, not
   arithmetic.
2. **Prefill is compute-bound; decode is memory-bound.** Derive it from intensity, not intuition.
3. **Decode needs batch ~300 to become compute-bound** on current hardware. Serving systems exist
   largely to reach that.
4. **Below the crossover, batching costs almost no latency** and multiplies throughput.
5. **`6ND` is the training-compute formula** behind every scaling-law calculation.
6. **Optimizer state, not weights, is why training does not fit.** 16–18 bytes per parameter.
7. **KV cache caps batch size, which caps throughput.** That chain is why cache management is a
   throughput optimization.
8. **Report MBU for decode, MFU for training.** Using the wrong one hides the real bottleneck.

### Self-check

- Compute the machine balance of an A100 and explain what the number means physically.
- Why is a matrix-vector product memory-bound while a matrix-matrix product is not?
- Derive prefill's and decode's arithmetic intensity. Why do they differ so much?
- At what batch size does decode become compute-bound on an H100? Show the arithmetic.
- Why is training `6ND` rather than `2ND`?
- Why does speculative decoding help decode but not prefill?
- An 8B model, bf16, AdamW: how much memory before activations? Which component dominates?
- Your decode MFU is 2%. Is that a problem? What should you measure instead?
- Why does KV cache size limit throughput, not just context length?

### What's next

Notebook 22 applies this to **distributed training** — how the memory budget above forces
parallelism, and how to choose among data, tensor, pipeline, sequence, and expert parallelism.
Notebook 23 applies it to **serving**.

### References

- Williams et al., 2009 — [Roofline: An Insightful Visual Performance Model](https://dl.acm.org/doi/10.1145/1498765.1498785)
- Kaplan et al., 2020 — [Scaling Laws for Neural Language Models](https://arxiv.org/abs/2001.08361) (Appendix: the FLOP accounting)
- Hoffmann et al., 2022 — [Chinchilla](https://arxiv.org/abs/2203.15556)
- Chowdhery et al., 2022 — [PaLM](https://arxiv.org/abs/2204.02311) (introduces MFU)
- Pope et al., 2022 — [Efficiently Scaling Transformer Inference](https://arxiv.org/abs/2211.05102) — the definitive treatment of this notebook's topic
- Dao et al., 2022 — [FlashAttention](https://arxiv.org/abs/2205.14135) (IO-awareness)
- Leviathan et al., 2022 — [Fast Inference via Speculative Decoding](https://arxiv.org/abs/2211.17192)
- Korthikanti et al., 2022 — [Reducing Activation Recomputation](https://arxiv.org/abs/2205.05198)
- Databricks — [LLM Inference Performance Engineering](https://www.databricks.com/blog/llm-inference-performance-engineering-best-practices) (MBU)